# Завдання 2 · Частина 4 — Емпіричний рівень помилки першого роду

За допомогою статистичного моделювання Монте-Карло перевірте, чи досягається задана теоретична ймовірність помилки першого роду (розмір критерію $\alpha$) для двостороннього тесту щодо середнього значення.

Джерело: [assignment_2.pdf](../input/assignment_2.pdf) (в оригіналі — «Задача 4. Інференційна статистика»).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

TASK_DIR = Path.cwd()
if (TASK_DIR / "labs" / "task2").is_dir():
    TASK_DIR = TASK_DIR / "labs" / "task2"
elif TASK_DIR.name == "notebooks":
    TASK_DIR = TASK_DIR.parent
INPUT_DIR = TASK_DIR / "input"
PDF_DIR = TASK_DIR / "pdf"
FIGURES_DIR = PDF_DIR / "figures"
SEED = 42
ALPHA = 0.05
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid")

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = PDF_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight"})


In [2]:
NULL_MEAN = 100
TRUE_VARIANCE = 25
TRUE_SD = np.sqrt(TRUE_VARIANCE)
N = 30
REPLICATIONS = 5000


## 4(а). Двосторонній тест для однієї вибірки

Змоделюйте вибірку обсягу $n=30$ із нормального розподілу з $\mu_0=100$ і $\sigma^2=25$. Перевірте $H_0:\mu=100$ проти двосторонньої альтернативи, вважаючи генеральну дисперсію невідомою. Вручну обчисліть тестову статистику, визначте аналітичну критичну область для $\alpha=0.05$ і прийміть рішення.

In [3]:
sample = rng.normal(loc=NULL_MEAN, scale=TRUE_SD, size=N)
sample_mean, sample_sd = sample.mean(), sample.std(ddof=1)
t_statistic = (sample_mean - NULL_MEAN) / (sample_sd / np.sqrt(N))
critical = stats.t.ppf(1 - ALPHA / 2, df=N - 1)
p_value = 2 * stats.t.sf(abs(t_statistic), df=N - 1)
single_result = pd.DataFrame({"n": [N], "mean": [sample_mean], "sd": [sample_sd],
    "t": [t_statistic], "df": [N - 1], "critical": [critical], "p": [p_value],
    "reject": [abs(t_statistic) > critical]})
display(single_result)
single_result.to_csv(RESULTS_DIR / "part4_single.csv", index=False)

,n,mean,sd,t,df,critical,p,reject
0,30,100.084071,3.883745,0.118564,29,2.04523,0.906438,False


***Answer***

Моделюємо $X_i\sim N(100,25)$, тобто генератор використовує стандартне відхилення 5. У тесті дисперсію вважаємо невідомою. Гіпотези: $H_0:\mu=100$, $H_1:\mu\ne100$.
$$t=\frac{\bar x-100}{s/\sqrt{30}}=\frac{100.084071-100}{3.883745/\sqrt{30}}=0.118564.$$
За незалежності та нормальності при $H_0$ маємо $T\sim t_{29}$. Критична область рівня 0.05:
$$(-\infty,-t_{0.975,29})\cup(t_{0.975,29},\infty),\qquad |t|>2.045230.$$
Отримано $p=0.906438$, тому $H_0$ не відхиляємо. У цій вибірці не виявлено статистично значущої відмінності середнього від 100. Невідхилення не є доведенням рівності.

## 4(б). Емпіричний рівень значущості

Незалежно змоделюйте $M=5000$ вибірок обсягу $n=30$ із $\mu_0=100$ і $\sigma^2=25$. Для кожної виконайте тест і збережіть індикатор $p_i$: 1 при відхиленні $H_0$, 0 в іншому випадку. Обчисліть $\widehat\alpha=M^{-1}\sum_{i=1}^{M}p_i$ та порівняйте з $\alpha=0.05$. Поясніть, чи очікуєте велику різницю, з огляду на відповідність умов експерименту припущенням критерію Стьюдента.

***Answer***

Перед моделюванням очікую невелику випадкову різницю між емпіричним рівнем та $0.05$: дані незалежні й нормальні, тому умови точного t-критерію виконано. Кількість повторень скінченна, тому точного збігу не очікую.

In [4]:
null_samples = rng.normal(loc=NULL_MEAN, scale=TRUE_SD, size=(REPLICATIONS, N))
t_statistics = (null_samples.mean(axis=1) - NULL_MEAN) / (null_samples.std(axis=1, ddof=1) / np.sqrt(N))
rejection_indicators = (np.abs(t_statistics) > critical).astype(int)
empirical_alpha = rejection_indicators.mean()
mc_se = np.sqrt(ALPHA * (1 - ALPHA) / REPLICATIONS)
size_results = pd.DataFrame({"M": [REPLICATIONS], "rejections": [rejection_indicators.sum()],
    "empirical_alpha": [empirical_alpha], "nominal_alpha": [ALPHA],
    "mc_se_under_null": [mc_se], "difference_in_se": [(empirical_alpha - ALPHA) / mc_se]})
display(size_results.T)
size_results.to_csv(RESULTS_DIR / "part4_size.csv", index=False)
pd.DataFrame({"t": t_statistics, "reject": rejection_indicators}).to_csv(
    RESULTS_DIR / "part4_replications.csv", index=False)

,0
M,5000.000000
rejections,266.000000
empirical_alpha,0.053200
nominal_alpha,0.050000
mc_se_under_null,0.003082
difference_in_se,1.038217


***Answer***

Для кожної з $M=5000$ незалежних вибірок обсягу 30 обчислено t-статистику та індикатор $I_i=\mathbf{1}\{|T_i|>t_{0.975,29}\}$. Ці індикатори відповідають позначенню $p_i$ в умові; вони не є p-значеннями.
$$\widehat\alpha=\frac1M\sum_{i=1}^M I_i=\frac{266}{5000}=0.0532.$$
Порівняно з 0.05 різниця становить 0.0032, тобто 0.32 відсоткового пункту. Для правильно відкаліброваного критерію стандартна похибка моделювання дорівнює
$$\operatorname{SE}_{MC}=\sqrt{\frac{0.05(1-0.05)}{5000}}\approx0.003082.$$
Спостережена різниця дорівнює приблизно 1.04 такої похибки, тому вона узгоджується зі звичайними випадковими коливаннями. Прогноз перед симуляцією підтвердився: нормальність, незалежність і оцінювання дисперсії з дільником $n-1$ забезпечують точний t-критерій, а скінченна кількість повторень дає невелике відхилення емпіричної частки від 0.05.